### The influence of tides in TNBD and DITD Mooring Data 2018 - 2019 

In [ ]:
import xarray as xr 
import numpy as np  
import matplotlib.pyplot as plt
import xrft 
import cmocean
import scipy
from scipy.signal import windows
import numpy as np

In [ ]:
# FOR LOADING ON MAC 
# TNBD Data - velocity 
tnbd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/vel_1100.nc')
tnbd_vel1000 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/vel_1100.nc')
# TNBD Data - SBE 
tnbd_sbe1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/sbe_1100.nc')
tnbd_sbe1000 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/sbe_1000.nc')
# DITD Data - velocity 
ditd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
ditd_vel1200 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/vel_1200.nc')
# DITD Data - SBE 
ditd_sbe1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/sbe_1100.nc')
ditd_sbe1200 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/sbe_1200.nc')

# FOR LOADING ON DELL 
# TNBD Data - velocity 
# tnbd_vel1100 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/TNB2018_Raw/TNBD/vel_1100.nc')
# tnbd_vel1000 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/TNB2018_Raw/TNBD/vel_1000.nc')
# # TNBD Data - SBE 
# tnbd_sbe1100 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/TNB2018_Raw/TNBD/sbe_1100.nc')
# tnbd_sbe1000 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/TNB2018_Raw/TNBD/sbe_1000.nc')
# # DITD Data - velocity 
# ditd_vel1100 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
# ditd_vel1200 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1200.nc')
# # DITD Data - SBE 
# ditd_sbe1100 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/sbe_1100.nc')
# ditd_sbe1200 = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/sbe_1200.nc')

In [ ]:
# plot a time series of the velocity data 
fig = plt.figure(figsize=(12,12),constrained_layout=True)
plt.subplot(4,1,1)
plt.plot(tnbd_vel1000.time,tnbd_vel1000.east)

plt.subplot(4,1,2)
plt.plot(tnbd_vel1100.time,tnbd_vel1100.east)

plt.subplot(4,1,3)
plt.plot(ditd_vel1100.time,ditd_vel1100.east)

plt.subplot(4,1,4)
plt.plot(ditd_vel1200.time,ditd_vel1200.east)



In [ ]:
# we need to remove the end of the data because dt gets messed up 
tnbd_vel1100.time.diff('time').plot()
ditd_vel1100.time.diff('time').plot()

In [ ]:
# crop to common date range 
tnbd_vel1100 = tnbd_vel1100.sel(time=slice('2018-03','2018-12-31'))
tnbd_sbe1100 = tnbd_sbe1100.sel(time=slice('2018-03','2018-12-31'))

In [ ]:
# crop to common date range 
ditd_vel1100 = ditd_vel1100.sel(time=slice('2018-03-13','2018-12-31'))
ditd_sbe1100 = ditd_sbe1100.sel(time=slice('2018-03-13','2018-12-31'))

Note that DITD velocity data is sampled every 10 minutes. TNBD velocity data is every 15 minutes. Will need to interpolate DITD data to same time array as TNBD when doing analysis later. Ok for now while dealing with the datasets seperately

In [ ]:
# dt is now the same 
tnbd_vel1100.east.time.diff('time').plot()

In [ ]:
# we can see that we have an even number of datapoints - good for FFT 
tnbd_vel1100.east

In [ ]:
ditd_vel1100.east

### Spectra using XRFT 

In [ ]:
# detrend data 
ditd_east = xrft.detrend(ditd_vel1100.east, dim='time')
ditd_north = xrft.detrend(ditd_vel1100.north, dim='time')

tnbd_east = xrft.detrend(tnbd_vel1100.east, dim='time')
tnbd_north = xrft.detrend(tnbd_vel1100.north, dim='time')


In [ ]:
# detrend data 
# east_detrended= xrft.detrend(tnbd_vel1100.east, dim='time')

In [ ]:
# calculate spectra 
ditd_east_spectra = xrft.power_spectrum(da=ditd_east.drop_vars(['index']),dim='time')
ditd_north_spectra = xrft.power_spectrum(da=ditd_north.drop_vars(['index']),dim='time')
tnbd_east_spectra = xrft.power_spectrum(da=tnbd_east.drop_vars(['index']),dim='time')
tnbd_north_spectra = xrft.power_spectrum(da=tnbd_north.drop_vars(['index']),dim='time')

In [ ]:
# spectra = xrft.power_spectrum(da=east_detrended.drop(['index']),dim='time')

In [ ]:
# take only positive part of spectrum 
tnbd_north_spectra = tnbd_north_spectra.where(tnbd_north_spectra.freq_time > 0, drop=True)
tnbd_east_spectra = tnbd_east_spectra.where(tnbd_east_spectra.freq_time > 0, drop=True)
ditd_north_spectra = ditd_north_spectra.where(ditd_north_spectra.freq_time > 0, drop=True)
ditd_east_spectra = ditd_east_spectra.where(ditd_east_spectra.freq_time > 0, drop=True)

In [ ]:
# spectra = spectra.where(spectra.freq_time > 0, drop=True)

In [ ]:
sampling_rate = 1/600  # 1/minutes 
nyquist = sampling_rate / 2

In [ ]:
nyquist

In [ ]:
1/nyquist

In [ ]:
nyquist/3600

In [ ]:
# output from xrft is in 1/s --> can tell by calculating & comparing with nyquist frequency 
tnbd_north_spectra.freq_time.max()

In [ ]:
ditd_east_spectra.freq_time.max()

In [ ]:
# nyquist freq is sampling rate / 2
tnbd_vel1100.east.time.diff(dim='time')

In [ ]:
fig = plt.figure(figsize=(12,8),constrained_layout=True)
plt.subplot(2,2,1)
plt.loglog(tnbd_east_spectra.freq_time*3600,tnbd_east_spectra)
plt.ylabel(r'Power Spectral Density [m$^2$/s$^2$/hr]')
# plt.xlabel('Frequency [1/hr]')
plt.title('TNBD Zonal Velocity, 1100 m')

plt.subplot(2,2,2)
plt.loglog(tnbd_north_spectra.freq_time*3600,tnbd_north_spectra)
# plt.ylabel(r'Power Spectral Density [m$^2$/s$^2$/hr]')
# plt.xlabel('Frequency [1/hr]')
plt.title('TNBD Meridional Velocity, 1100 m')

plt.subplot(2,2,3)
plt.loglog(ditd_east_spectra.freq_time*3600,ditd_east_spectra)
plt.ylabel(r'Power Spectral Density [m$^2$/s$^2$/hr]')
plt.xlabel('Frequency [1/hr]')
plt.title('DITD Zonal Velocity, 1100 m')

plt.subplot(2,2,4)
plt.loglog(ditd_north_spectra.freq_time*3600,ditd_north_spectra)
# plt.ylabel(r'Power Spectral Density [m$^2$/s$^2$/hr]')
plt.xlabel('Frequency [1/hr]')
plt.title('DITD Meridional Velocity, 1100 m')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Frequency and PSD (assumed given)
freq = spectra.freq_time * 3600  # Frequency in 1/hr
psd = spectra

# Create base plot
fig, ax = plt.subplots(figsize=(10,6))
ax.loglog(freq, psd, label='PSD')
ax.set_xlabel('Frequency [1/hr]')
ax.set_ylabel(r'Power Spectral Density [m$^2$/s$^2$/hr]')

# Add vertical shaded regions for tidal peaks
tidal_peaks = {
    'M2': 0.0805,
    'S2': 0.0833,
    'K1': 0.0418,
    'O1': 0.0387
}
# peak_width = 0.002  # Width of shaded region

# for name, f in tidal_peaks.items():
#     ax.axvspan(f - peak_width, f + peak_width, color='gray', alpha=0.3)
#     ax.text(f, ax.get_ylim()[1]*0.8, name, rotation=90,
#             ha='center', va='top', fontsize=9, color='gray')

for name, f in tidal_peaks.items():
    ax.axvline(f, color='gray', linestyle='--', linewidth=1)
    ax.text(f, ax.get_ylim()[1]*0.8, name, rotation=90,
            ha='center', va='top', fontsize=9, color='gray')
    
# Create top x-axis with period
ax_top = ax.twiny()
ax_top.set_xscale('log')

freq_limits = ax.get_xlim()
freq_limits = [spectra.freq_time[0]*3600, spectra.freq_time[-1]*3600]
period_limits = [1/freq_limits[0], 1/freq_limits[1]]
ax_top.set_xlim(period_limits)
ax_top.set_xlabel('Period [hr]')
ax_top.xaxis.set_ticks_position('top')
ax_top.xaxis.set_label_position('top')
# ax_top.ticklabel_format(style='plain', axis='x')

plt.tight_layout()
plt.show()


### Filter out tides 

In this section the goal is to filter out the diurnal tidal signal through the use of a cosine filtering window. 

In [ ]:
sampling_hours = 0.25  # sampling period (hrs)
window_days = 3     # desired window 
window_size = int(window_days * 24 / sampling_hours)   # window size based on sampling period 


In [ ]:
window_size

In [ ]:
if window_size % 2 == 0:
    window_size += 1

In [ ]:
from scipy.signal import windows
import numpy as np

cosine_window = windows.cosine(window_size)
cosine_window /= cosine_window.sum()  # normalize so it acts like a moving average


In [ ]:
len(cosine_window)

In [ ]:
sampling_hours = 0.25  # sampling period (hrs)
window_days = 3     # desired window 
window_size = int(window_days * 24 / sampling_hours) 

if window_size % 2 == 0:
    window_size += 1

cosine_window = windows.cosine(window_size)
cosine_window /= cosine_window.sum()  # normalize so it acts like a moving average

# assume velocity_da is your DataArray with dimension "time"
cosine_weights = xr.DataArray(cosine_window, dims="window")

# create rolling window and apply weights
tnbd_east_filtered = (
    tnbd_vel1100.east
    .rolling(time=window_size, center=True)
    .construct("window")
    .dot(cosine_weights)
)

tnbd_north_filtered = (
    tnbd_vel1100.north
    .rolling(time=window_size, center=True)
    .construct("window")
    .dot(cosine_weights)
)


In [ ]:
sampling_hours = 10/60  # sampling period (hrs)
window_days = 3     # desired window 
window_size = int(window_days * 24 / sampling_hours) 

if window_size % 2 == 0:
    window_size += 1

cosine_window = windows.cosine(window_size)
cosine_window /= cosine_window.sum()  # normalize so it acts like a moving average

# assume velocity_da is your DataArray with dimension "time"
cosine_weights = xr.DataArray(cosine_window, dims="window")

# create rolling window and apply weights
ditd_east_filtered = (
    ditd_vel1100.east
    .rolling(time=window_size, center=True)
    .construct("window")
    .dot(cosine_weights)
)

ditd_north_filtered = (
    ditd_vel1100.north
    .rolling(time=window_size, center=True)
    .construct("window")
    .dot(cosine_weights)
)

In [ ]:
fig = plt.figure(figsize=(12,8))
plt.subplot(4,1,1)
plt.plot(tnbd_vel1100.time,tnbd_vel1100.east,color='tab:orange',label='In-situ velocity')
plt.plot(tnbd_east_filtered.time,tnbd_east_filtered,label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('TNBD Zonal')
plt.legend()
plt.xticks([])

plt.subplot(4,1,2)
plt.plot(tnbd_vel1100.time,tnbd_vel1100.north,color='tab:orange',label='In-situ velocity')
plt.plot(tnbd_north_filtered.time,tnbd_north_filtered,label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('TNBD Meridional')
plt.xticks([])

plt.subplot(4,1,3)
plt.plot(ditd_vel1100.time,ditd_vel1100.east,color='tab:orange',label='In-situ velocity')
plt.plot(ditd_east_filtered.time,ditd_east_filtered,label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.xticks([])
plt.title('DITD Zonal')

plt.subplot(4,1,4)
plt.plot(ditd_vel1100.time,ditd_vel1100.north,color='tab:orange',label='In-situ velocity')
plt.plot(ditd_north_filtered.time,ditd_north_filtered,label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('DITD Meridional')

In [ ]:
# get tidal velocity by subtracting filtered from observed 
tnbd_east_tidal = tnbd_vel1100.east - tnbd_east_filtered
tnbd_north_tidal = tnbd_vel1100.north - tnbd_north_filtered
ditd_east_tidal = ditd_vel1100.east - ditd_east_filtered
ditd_north_tidal = ditd_vel1100.north - ditd_north_filtered


In [ ]:
(2880/15)/60

In [ ]:
1440*3/10

In [ ]:
# Plotting with a 3 day rolling average 
fig = plt.figure(figsize=(12,8))
plt.subplot(4,1,1)
# plt.plot(tnbd_vel1100.time,tnbd_vel1100.east,color='tab:orange',label='In-situ velocity')
plt.plot(tnbd_east_tidal.rolling(time=288).mean().time,tnbd_east_tidal.rolling(time=288).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('TNBD Zonal Tides')
plt.legend()
plt.xticks([])

plt.subplot(4,1,2)
# plt.plot(tnbd_vel1100.time,tnbd_vel1100.north,color='tab:orange',label='In-situ velocity')
plt.plot(tnbd_north_tidal.rolling(time=288).mean().time,tnbd_north_tidal.rolling(time=288).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('TNBD Meridional Tides')
plt.xticks([])

plt.subplot(4,1,3)
# plt.plot(ditd_vel1100.time,ditd_vel1100.east,color='tab:orange',label='In-situ velocity')
plt.plot(ditd_east_tidal.rolling(time=432).mean().time,ditd_east_tidal.rolling(time=432).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.xticks([])
plt.title('DITD Zonal Tides')

plt.subplot(4,1,4)
# plt.plot(ditd_vel1100.time,ditd_vel1100.north,color='tab:orange',label='In-situ velocity')
plt.plot(ditd_north_tidal.rolling(time=432).mean().time,ditd_north_tidal.rolling(time=432).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('DITD Meridional Tides')

In [ ]:
# subtract tides from observed velocity 
tnbd_east_detided = tnbd_vel1100.east - tnbd_east_tidal 
tnbd_north_detided = tnbd_vel1100.north - tnbd_north_tidal 

ditd_east_detided = ditd_vel1100.east - ditd_east_tidal 
ditd_north_detided = ditd_vel1100.north - ditd_north_tidal 

In [ ]:
# Plotting with a 3 day rolling average 
fig = plt.figure(figsize=(12,8))
plt.subplot(4,1,1)
# plt.plot(tnbd_vel1100.time,tnbd_vel1100.east,color='tab:orange',label='In-situ velocity')
plt.plot(tnbd_east_detided.rolling(time=288).mean().time,tnbd_east_detided.rolling(time=288).mean(),label='Filtered velocity')
# plt.plot(tnbd_east_filtered.rolling(time=288).mean().time,tnbd_east_filtered.rolling(time=288).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('TNBD Zonal Velocity - Tides Removed')
# plt.legend()
plt.xticks([])

plt.subplot(4,1,2)
# plt.plot(tnbd_vel1100.time,tnbd_vel1100.north,color='tab:orange',label='In-situ velocity')
plt.plot(tnbd_north_detided.rolling(time=288).mean().time,tnbd_north_detided.rolling(time=288).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('TNBD Meridional Velocity - Tides Removed')
plt.xticks([])

plt.subplot(4,1,3)
# plt.plot(ditd_vel1100.time,ditd_vel1100.east,color='tab:orange',label='In-situ velocity')
plt.plot(ditd_east_detided.rolling(time=432).mean().time,ditd_east_detided.rolling(time=432).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.xticks([])
plt.title('DITD Zonal Velocity - Tides Removed')

plt.subplot(4,1,4)
# plt.plot(ditd_vel1100.time,ditd_vel1100.north,color='tab:orange',label='In-situ velocity')
plt.plot(ditd_north_detided.rolling(time=432).mean().time,ditd_north_detided.rolling(time=432).mean(),label='Filtered velocity')
plt.ylabel('speed (m/s)')
plt.title('DITD Meridional Velocity - Tides Removed')

In [ ]:
# calculate velocity magnitude 
tnbd_vel_mag = np.sqrt(tnbd_east_detided**2 + tnbd_north_detided**2)
ditd_vel_mag = np.sqrt(ditd_east_detided**2 + ditd_north_detided**2)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(tnbd_vel_mag.time,tnbd_vel_mag,label='TNBD')
plt.plot(ditd_vel_mag.time,ditd_vel_mag,label='DITD')
plt.legend()
plt.ylabel('Speed (m/s)')
plt.title('Velocity Magnitude at TNBD and DITD - Detided')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(tnbd_vel_mag.sel(time=slice('2018-09-01','2018-09-30')).time,tnbd_vel_mag.sel(time=slice('2018-09-01','2018-09-30')),label='TNBD')
plt.plot(ditd_vel_mag.sel(time=slice('2018-09-01','2018-09-30')).time,ditd_vel_mag.sel(time=slice('2018-09-01','2018-09-30')),label='DITD')
plt.legend()
plt.ylabel('Speed (m/s)')
plt.title('Velocity Magnitude at TNBD and DITD - Detided')

In [ ]:
# fig = plt.figure(figsize=(12,4))
# plt.plot(tidal.rolling(time=2880).mean().time,tidal.rolling(time=2880).mean())
# plt.plot(tnbd_sbe1100.rolling(time=

In [ ]:
# plt.plot(tidal.resample(time='30D').mean().time,tidal.resample(time='30D').mean())

### Compare tides with SBE data 

In [ ]:
# calculate tidal magnitude 
ditd_tides = np.sqrt(ditd_east_tidal**2 + ditd_north_tidal**2)
tnbd_tides = np.sqrt(tnbd_east_tidal**2 + tnbd_north_tidal**2)


In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(ditd_tides.rolling(time=432).mean().time,ditd_tides.rolling(time=432).mean(),label='DITD')
plt.plot(tnbd_tides.rolling(time=288).mean().time,tnbd_tides.rolling(time=288).mean(),label='TNBD')
plt.legend()
plt.title('Weekly Rolling Average Tides at TNBD and DITD')
plt.ylabel('Speed (m/s)')

In [ ]:
ditd_temp_weekly = ditd_sbe1100.rolling(time=10080*4).mean().temp

In [ ]:
10080/3

In [ ]:
tnbd_temp_weekly = tnbd_sbe1100.rolling(time=3360*4).mean().temp

In [ ]:
10/60/24/30

In [ ]:
ditd_tides

In [ ]:
# 10080
fig, ax = plt.subplots(figsize=(12,4))
line, = ax.plot(ditd_tides.rolling(time=10080*4).mean().time,ditd_tides.rolling(time=10080*4).mean(),label='Tides')
ax.set_ylabel('Speed (m/s)')

# twin x
ax1 = ax.twinx()
line1, = ax1.plot(ditd_temp_weekly.time,ditd_temp_weekly,color='tab:red',label='Temp')
ax1.set_ylabel('Temp')

ax.set_title('Tides and Temp at DITD')

lines = [line, line1]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels)


In [ ]:
# 10080
fig, ax = plt.subplots(figsize=(12,4))
line, = ax.plot(tnbd_tides.rolling(time=288).mean().time,tnbd_tides.rolling(time=288).mean(),label='Tides')
ax.set_ylabel('Speed (m/s)')

# twin x
ax1 = ax.twinx()
line1, = ax1.plot(tnbd_temp_weekly.time,tnbd_temp_weekly,color='tab:red',label='Temp')
ax1.set_ylabel('Temp')

ax.set_title('Tides and Temp at TNBD')

lines = [line, line1]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels)

### FFT Analysis via numpy 

In [ ]:
fh = scipy.fft.rfft(east_detrend)

In [ ]:
freq = scipy.fft.rfftfreq(n=len(east_detrend),d=15)

In [ ]:
len(freq)

In [ ]:
len(fh)

In [ ]:
plt.plot(freq,fh)

In [ ]:
magnitude_spectrum = np.abs(fh)

In [ ]:
len(magnitude_spectrum )

In [ ]:
plt.loglog(freq,magnitude_spectrum)
# plt.xlim(-0.00001,0.005)
plt.ylim(10e-4,10e3)

In [ ]:
from scipy.signal import butter, lfilter

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

In [ ]:
# Sample rate and desired cutoff frequencies (in Hz).
fs = 1/15
lowcut = 0.0006
highcut = 0.0008


In [ ]:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.signal import freqz

In [ ]:
# Plot the frequency response for a few different orders.
plt.figure(1)
plt.clf()
for order in [3, 6, 9]:
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    w, h = freqz(b, a, worN=2000)
    plt.plot((fs * 0.5 / np.pi) * w, abs(h), label="order = %d" % order)

plt.plot([0, 0.5 * fs], [np.sqrt(0.5), np.sqrt(0.5)],
         '--', label='sqrt(0.5)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Gain')
plt.grid(True)
plt.legend(loc='best')

In [ ]:
y = butter_bandpass_filter(east_detrend, lowcut, highcut, fs, order=3)

In [ ]:
y

In [ ]:
east_detrend[0:30]

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(east.time,east_detrend,label='Linearly detrended east velocity')
plt.plot(east.time,y,label='Tidal Frequency?')
# plt.plot(east.isel(time=slice(0,1000)).time,(east_detrend - y)[0:1000],label='velocity - tidal frequency')
plt.legend()

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(east.isel(time=slice(0,1000)).time,east_detrend[0:1000],label='Linearly detrended east velocity')
plt.plot(east.isel(time=slice(0,1000)).time,y[0:1000],label='Tidal Frequency?')
plt.plot(east.isel(time=slice(0,1000)).time,(east_detrend - y)[0:1000],label='velocity - tidal frequency')
plt.legend()

In [ ]:
plt.plot(east.time,(east_detrend - y))

In [ ]:
# east.time.diff(dim='time')[0].values.astype('float64')

In [ ]:
inverse_sample_rate = 1/east.time.diff(dim='time')[0].values.astype('float64')

In [ ]:
inverse_sample_rate

In [ ]:
fh = np.fft.rfft(east_detrend)

In [ ]:
len(fh.real)

In [ ]:
#If n is even, the length of the transformed axis is (n/2)+1
len(fh)

In [ ]:
dt = 15
N = len(east_detrend)
scale = 1/(N*dt)

In [ ]:
N*dt

In [ ]:
scale

In [ ]:
delta_f = scale 

In [ ]:
freq = delta_f * np.arange(N)

In [ ]:
plt.plot(freq)

In [ ]:
# only positive 
freq_pos = delta_f * np.arange(((N/2)+1))

In [ ]:
len(freq_pos)

In [ ]:
plt.plot(freq_pos,fh.real)

In [ ]:
plt.plot(freq_pos)

In [ ]:
fh = 

In [ ]:
freq*scale

In [ ]:
# calculate frequencies associated with rfft 
freq = np.fft.rfftfreq(n=len(east_detrend),d=1/15)

In [ ]:
# length matches that of fft 
len(freq)

In [ ]:
plt.plot(freq*scale)

In [ ]:
(freq*scale).max()

In [ ]:
(freq*scale).min()

In [ ]:
time_strings = east.time.values

In [ ]:
time_strings

In [ ]:
T = (time_strings[-1] - time_strings[0]).astype('timedelta64[m]')

In [ ]:
T

In [ ]:
T = 426225  #minutes 

In [ ]:
delta_t = 15   # minutes 

In [ ]:
sample_intervals = T/delta_t

In [ ]:
sample_intervals

In [ ]:
# sample points 
N = sample_intervals + 1 

In [ ]:
N

In [ ]:
nyquist_freq = 1/(2*delta_t)

In [ ]:
nyquist_freq

In [ ]:
(sample_intervals/2)/(sample_intervals*delta_t)

In [ ]:
# our max freq is nyquist frequency 
# our min freq is 1/Time_length 
1/T

In [ ]:
freq

In [ ]:
N = len(east_detrend)
dt = 1/15
# define scaling factor 
scale = 1/(N*dt)

In [ ]:
# get spectrum by taking the real part of FFT 
spec = 2*(fh*fh.conj()).real / scale / N**2

# account for zeroth frequency 
spec[0] = spec[0]/2

In [ ]:
len(spec)

In [ ]:
plt.plot(freq*sacle,spec)

In [ ]:
u_dft = xrft.dft(east, true_phase=True, true_amplitude=True, detrend='constant',window='hamming')

In [ ]:
u_dft_real = u_dft.real.where(u_dft.real > 0, drop=True)
# u_dft_real = u_dft.real.where(u_dft.real > 0, drop=True)

In [ ]:
u_dft_real = u_dft_real.where(u_dft.freq_time > 0, drop=True)

In [ ]:
u_dft_real['freq_time'] = u_dft_real.freq_time*(60/1)

In [ ]:
u_dft_real

In [ ]:
fig = plt.figure(figsize=(12,4))
(u_dft_real).plot()
plt.xlim(0,0.005)

In [ ]:
# Compute the PSD
psd = xrft.power_spectrum(east, dim='time')

# Plot the PSD
plt.figure(figsize=(12, 4))
plt.loglog(psd.freq_time, psd)
plt.xlabel('Frequency')
plt.ylabel('Power Spectral Density')
plt.title('Power Spectrum')
plt.grid(True)
plt.show()

## using my functions

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from scipy import stats
from scipy import signal
from numpy.fft import fft, ifft
from numpy import genfromtxt
from scipy.stats import binned_statistic

def hanning(data,N):
    """
    Create a hanning window filter
    
    Inputs: data - an array (1D timeseries or spatial data),
            N - length of data
    
    Output: data - data array with hanning window applied 
    """
    
    win = np.hanning(N)
    win = np.sqrt(N/(win**2).sum())*win     #why do we have to do this step? 
    data *= win
    
    return data

def calc_freq(N,dt):
    """
    Calculate the frequency of a given dataset
    
    Input: N - number of data points (length of dataset)
           dt - resolution of dataset 
           
    Output: freq - calculated frequency
            scale - scaling factor for frequency
    """
    
    # figure out if data length is even or odd
    # %2 gives you remainder (i.e. if N%2 = 0 then N is even)
    if N%2:
        even = False
    else:
        even = True
        
    # define scaling factor 
    scale = 1/(N*dt)
    
    # get frequency 
    if even:
        freq = np.arange(N/2+1) * scale
    else:
        freq = np.arange((N-1)/2 +1) * scale
        
    return freq, scale

def calc_spectrum(data,scale,N):
    """
    Calculate spectrum using FFT 
    
    Input: data - 1D timeseries or spatial data,
           scale - scaling factor (1/(N*dt))
           N - number of data points (length of data series)
           
    Output: spec - calculated spectrum 
    """
    # calculate FFT of dataset 
    fh = np.fft.rfft(data)
    
    # get spectrum by taking the real part of FFT 
    spec = 2*(fh*fh.conj()).real / scale / N**2
    
    # account for zeroth frequency 
    spec[0] = spec[0]/2
    
    if N%2:
        spec[-1] = spec[-1]/2
        
    return spec
    
    
def spec_var(scale,spec):
    """
    Calculate the variance of the spectrum 
    
    Input: scale - (1/(N*dt))
           spec - spectrum 
           
    Output: spec_var - spectrum variance
    
    """
    
    spec_var = scale*spec[1:].sum()
    
    return spec_var

def get_spectrum(dataset,dt,window=False):
    """
    Calculate the spectrum of a 1D timeseries / spatial series using FFT 
    
    Input: dataset - 1D data series (by time or distance)
    Output: freq - The frequency of the dataset,
            spec - spectrum of the dataset 
    """
    
    data = dataset.copy()
    data = signal.detrend(data)
    
    N = len(data)

    if window == True:
        data = hanning(data,N)
    
    freq,scale = calc_freq(N,dt)
    
    spec = calc_spectrum(data,scale,N)
    
    return freq, spec

In [ ]:
freq,spec = get_spectrum(east,15,window=True)

In [ ]:
plt.plot(freq,spec)
plt.xlim(0.0006,0.0008)

### starting from scratch with numpy functions 

In [ ]:
# step 1: ensure data is even 
# step 2: detrend the data 
tnbd_vel1100.east

In [ ]:
u_detrend = scipy.signal.detrend(tnbd_vel1100.east)

In [ ]:
np.fft.rfft(u_detrend)

In [ ]:
subset = tnbd_vel1100.east.sel(time=slice('2018-03','2019-01-02'))

In [ ]:
subset

In [ ]:
time_strings = subset.time.values

In [ ]:
time_strings

In [ ]:
T = (time_strings[-1] - time_strings[0]).astype('timedelta64[m]')

In [ ]:
T

In [ ]:
T = 426225  #minutes 

In [ ]:
delta_t = 15   # minutes 

In [ ]:
sample_intervals = T/delta_t

In [ ]:
sample_intervals

In [ ]:
# sample points 
N = sample_intervals + 1 

In [ ]:
N

In [ ]:
nyquist_freq = 1/(2*delta_t)

In [ ]:
nyquist_freq

In [ ]:
(sample_intervals/2)/(sample_intervals*delta_t)

In [ ]:
freq = np.fft.fftfreq(n=len(subset),d=1/15)

In [ ]:
plt.plot(freq)

In [ ]:
subset

In [ ]:
freq, spec = scipy.signal.periodogram(east_detrend,1/15,detrend='linear')

In [ ]:
plt.plot(freq)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(freq,spec)
plt.xlabel('Frequency [1/min]')
plt.ylabel(r'Power Spectral Density [m$^{2}$/s$^{2}$/(1/min)]')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(freq,spec)
plt.xlim(0,0.002)
plt.xlabel('Frequency [1/min]')
plt.ylabel(r'Power Spectral Density [m$^{2}$/s$^{2}$/(1/min)]')
# wavelet transform 

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(freq,spec)
plt.xlim(0.00062,0.00072)
plt.xlabel('Frequency [1/min]')
plt.ylabel(r'Power Spectral Density [m$^{2}$/s$^{2}$/(1/min)]')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(freq,spec)
plt.xlim(0.00130,0.00135)
plt.xlabel('Frequency [1/min]')
plt.ylabel(r'Power Spectral Density [m$^{2}$/s$^{2}$/(1/min)]')
plt.ylim(0,20)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.semilogx(freq,spec)
plt.xlabel('Frequency [1/min]')
plt.ylabel(r'Power Spectral Density [m$^{2}$/s$^{2}$/(1/min)]')
# plt.ylim(10e-8,10e2)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.loglog(freq,spec)
plt.xlabel('Frequency [1/min]')
plt.ylabel(r'Power Spectral Density [m$^{2}$/s$^{2}$/(1/min)]')
plt.ylim(10e-8,10e2)

In [ ]:
subset.values

In [ ]:
fft = scipy.fft.fft(subset.values,n=len(subset))

In [ ]:
fft 

In [ ]:
len(fft)

In [ ]:
fft_real = fft.conj().real

In [ ]:
len(fft_real)

In [ ]:
len(fft_real[fft_real>0])

In [ ]:
fft_real_positive = fft_real[np.where(fft_real >= 0)]

In [ ]:
len(fft_real_positive)

In [ ]:
fft_freq = scipy.fft.fftfreq(n=len(subset),d=1/15)

In [ ]:
len(fft_freq)

In [ ]:
fft_freq_positive = fft_freq[np.where(fft_freq >= 0)]

In [ ]:
len(fft_freq[fft_freq>0])

In [ ]:
len(fft_freq_positive)

In [ ]:
plt.plot(fft_freq,fft_real)